# 01 — EDA: Student Base

> **Notebook 01 of 7** | Learning Retention Analytics  
> First exploratory analysis: who are the students, what are their outcomes, and what does the data look like?

## Purpose and Scope

This notebook is the **first of two EDA (Exploratory Data Analysis) notebooks** in the project. It examines the **student base**: demographics, enrollment patterns, outcome distribution, and course landscape.

**What this notebook covers:**
- Dataset dimensions and scale
- Outcome distribution (4-class and binarized)
- Demographic profile of the student population
- Enrollment patterns (registration timing)
- Course-level overview (enrollment, completion, design characteristics)
- Data quality assessment
- Brief engagement baseline (preview)

**What comes next:**
- **Notebook 02** (`02_eda_engagement_patterns.ipynb`): detailed behavioral analysis — daily clickstream patterns, early engagement signals, temporal trends

**Connection to business questions:**  
This EDA does not answer BQ1–BQ5 directly. Instead, it establishes the population profile and data quality baseline that all subsequent analyses depend on. Think of it as *understanding the terrain before navigating*.

> **What is EDA?** Exploratory Data Analysis is the practice of examining a dataset before formal modeling or hypothesis testing. The goal is to discover patterns, spot anomalies, check assumptions, and build intuition about the data. For a deeper introduction, see [`docs/eda-guide.md`](../docs/eda-guide.md).

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [Dataset Overview — Shape and Scale](#2.-Dataset-Overview-—-Shape-and-Scale)
3. [Outcome Distribution](#3.-Outcome-Distribution)
4. [Outcome by Course](#4.-Outcome-by-Course)
5. [Demographic Profile](#5.-Demographic-Profile)
6. [Enrollment Patterns](#6.-Enrollment-Patterns)
7. [Course Landscape](#7.-Course-Landscape)
8. [Data Quality Assessment](#8.-Data-Quality-Assessment)
9. [Engagement Baseline — Preview](#9.-Engagement-Baseline-—-Preview)
10. [Key Takeaways and Next Steps](#10.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- Notebooks live in `notebooks/` but project modules are in `src/` at the project root. We add the project root to `sys.path` so that `from src.config import ...` works. The linter rule `E402` (imports not at top of file) is suppressed for notebooks in `pyproject.toml`.
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer. This returns a `pandas.DataFrame` and ensures we never call `duckdb.connect()` directly (see [ADR-003](../docs/ADR.md)).
- Figures are saved to `reports/figures/` at 150 DPI. Since `nbstripout` removes notebook outputs before commit, the saved PNGs are the persistent visual record.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# Adding the root to sys.path lets us import from src/ as if running from root.
# We search upward for pyproject.toml instead of assuming cwd is always notebooks/,
# so the notebook works regardless of where the kernel is launched from
# (JupyterLab, VS Code, Cursor, repo root, etc.).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Binary version: completed (1) vs not completed (0)
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: 'Completed', 0: 'Not completed'}
# Label-keyed palette for seaborn when x-axis uses mapped string categories
PALETTE_BINARY_LABELS = {'Completed': '#55A868', 'Not completed': '#C44E52'}
# Sequential palette for heatmaps and continuous scales
PALETTE_SEQUENTIAL = 'YlOrRd'
# Shared axis label — the unit of analysis is the enrollment, not the student
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


def query_demo_distribution(column: str) -> pd.DataFrame:
    """Query count and completion rate by a demographic column.

    Returns DataFrame with columns: [category, n, completion_rate].
    Column names are standardized so that plot_demo_completion() works
    regardless of which demographic variable we are analyzing.

    COALESCE handles NULL values (e.g. imd_band) — SQL NULLs become
    pandas NaN which matplotlib cannot use as categorical axis labels.
    Converting to 'Unknown' at the SQL level keeps downstream code clean.
    """
    return execute_query(f'''
        SELECT
            COALESCE(CAST({column} AS VARCHAR), 'Unknown') AS category,
            COUNT(*)                                        AS n,
            ROUND(100.0 * SUM(completed) / COUNT(*), 1)     AS completion_rate
        FROM v_student_enriched
        GROUP BY {column}
        ORDER BY n DESC
    ''')


def plot_demo_completion(
    df: pd.DataFrame,
    title: str,
    figname: str,
    horizontal: bool = False,
) -> None:
    """Bar chart with completion-rate annotations for a demographic variable.

    Parameters
    ----------
    df : DataFrame with columns [category, n, completion_rate]
    title : chart title
    figname : filename without extension (e.g. '01_demo_gender')
    horizontal : if True, horizontal bars (better for long category labels)
    """
    fig, ax = plt.subplots(figsize=FIG_SIZE)

    if horizontal:
        bars = ax.barh(df['category'], df['n'], color='#4C72B0', edgecolor='white')
        for bar, (_, row) in zip(bars, df.iterrows()):
            # Annotate each bar with the completion rate
            ax.text(
                bar.get_width() + ax.get_xlim()[1] * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f"{row['completion_rate']:.1f}% completed",
                va='center', fontsize=10, color='#333333',
            )
        ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
        ax.invert_yaxis()
    else:
        bars = ax.bar(df['category'], df['n'], color='#4C72B0', edgecolor='white')
        for bar, (_, row) in zip(bars, df.iterrows()):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                f"{row['completion_rate']:.1f}%",
                ha='center', fontsize=10, color='#333333',
            )
        ax.set_ylabel(LABEL_NUM_ENROLLMENTS)
        plt.xticks(rotation=45, ha='right')

    ax.set_title(title)
    sns.despine()
    fig.tight_layout()
    save_fig(fig, figname)
    plt.show()


# --- Prerequisite check ---
# Verify the database is populated before proceeding
try:
    _check = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _n_rows = _check['n'].iloc[0]
    if _n_rows == 0:
        raise RuntimeError('v_student_enriched is empty')
    print(f'Database OK — v_student_enriched has {_n_rows:,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query v_student_enriched. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Dataset Overview — Shape and Scale

The first step in any EDA is understanding **how much data we have** and **what it looks like at a high level**.

**Key concept: the unit of analysis.**  
In OULAD, one student can enroll in multiple courses (modules). The unit of analysis is the **student-module enrollment** — identified by the composite key `(id_student, code_module, code_presentation)` — not the unique student.

This means the total number of rows in `v_student_enriched` is the number of **enrollments**, which is larger than the number of unique students. A student who completes Course A but withdraws from Course B appears as two rows: one *completed* and one *not completed*. Both are valid data points for retention analysis.

In [ ]:
# --- Dataset dimensions ---
# How many enrollments, unique students, courses, and presentations?
df_dims = execute_query('''
    SELECT
        COUNT(*)                                                 AS n_enrollments,
        COUNT(DISTINCT id_student)                               AS n_unique_students,
        COUNT(DISTINCT code_module)                              AS n_modules,
        COUNT(DISTINCT code_module || '-' || code_presentation)  AS n_course_presentations
    FROM v_student_enriched
''')

n_enroll = df_dims['n_enrollments'].iloc[0]
n_students = df_dims['n_unique_students'].iloc[0]
n_modules = df_dims['n_modules'].iloc[0]
n_cp = df_dims['n_course_presentations'].iloc[0]

print('=== Dataset Dimensions ===')
print(f'  Total enrollments:       {n_enroll:>8,}')
print(f'  Unique students:         {n_students:>8,}')
print(f'  Modules (courses):       {n_modules:>8}')
print(f'  Course-presentations:    {n_cp:>8}')
print(f'\n  → On average, each student enrolls in {n_enroll / n_students:.1f} modules')

> **Reading the numbers:** The enrollment count is larger than the unique student count because some students enroll in multiple modules. This is normal in a university setting where students take several courses per semester.
>
> The distinction between *enrollments* and *students* matters throughout this analysis: all completion rates, demographic breakdowns, and engagement metrics are computed **per enrollment**, not per unique person.

In [ ]:
# --- Course summary table ---
# One row per course-presentation with key metrics from v_course_profile.
# We select all columns here because Section 7 (Course Landscape) reuses
# df_courses for scatter plots and correlation analysis.
df_courses = execute_query('''
    SELECT
        code_module,
        code_presentation,
        course_length_days,
        n_enrolled,
        n_completed,
        completion_rate_pct,
        n_withdrew,
        withdrawal_rate_pct,
        n_assessments,
        assessments_per_30_days,
        n_vle_resources,
        n_activity_types
    FROM v_course_profile
    ORDER BY code_module, code_presentation
''')

print(f'=== Course-Presentation Summary ({len(df_courses)} rows) ===\n')
df_courses

> **What this table tells us:**
> - Each row is a unique **course-presentation** (a specific module offered in a specific semester, e.g. AAA-2013J).
> - **Completion rate** varies across courses — some are notably higher or lower, which BQ4 will investigate.
> - **Course design** also varies: some courses have more assessments, others more VLE resources.
> - The `course_length_days` column shows that courses range from short to long — this affects withdrawal timing (BQ1).
>
> We will return to this table in [Section 7 (Course Landscape)](#7.-Course-Landscape) for visual analysis.

## 3. Outcome Distribution

The **target variable** in this project is `final_result`, which takes four values:

| Value | Meaning |
|-------|--------|
| **Pass** | Student completed the course and passed |
| **Distinction** | Student completed with honors |
| **Fail** | Student stayed enrolled but did not pass |
| **Withdrawn** | Student actively withdrew before the end |

For most analyses, we **binarize** this into:
- **Completed** (= Pass + Distinction) → `completed = 1`
- **Not completed** (= Fail + Withdrawn) → `completed = 0`

This binarization is documented in [ADR-006](../docs/ADR.md) and consistent with OULAD literature. The key distinction to keep in mind: **Withdrawn ≠ Fail**. Withdrawn students actively left (a behavioral signal); Failed students stayed but did not pass (an academic outcome). This distinction matters for intervention design (BQ5).

In [ ]:
# --- 4-class outcome distribution ---
df_outcome = execute_query('''
    SELECT
        final_result,
        COUNT(*) AS n_enrollments,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched
    GROUP BY final_result
    ORDER BY n_enrollments DESC
''')

print('=== Outcome Distribution (4-class) ===\n')
print(df_outcome.to_string(index=False))

# --- Horizontal bar chart ---
# Ordered from largest to smallest category for visual hierarchy
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_OUTCOME[r] for r in df_outcome['final_result']]
bars = ax.barh(df_outcome['final_result'], df_outcome['n_enrollments'], color=colors,
               edgecolor='white')

# Annotate each bar with count and percentage
for bar, (_, row) in zip(bars, df_outcome.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n_enrollments']):,} ({row['pct']:.1f}%)",
        va='center', fontsize=11,
    )

ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('How Did Students Finish? (4-class outcome)')
ax.invert_yaxis()  # largest at top
sns.despine(left=True)
fig.tight_layout()
save_fig(fig, '01_outcome_distribution_4class')
plt.show()

In [ ]:
# --- Binary outcome (completed vs not completed) ---
# This is the "headline number" for the project
df_binary = execute_query('''
    SELECT
        completed,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched
    GROUP BY completed
    ORDER BY completed DESC
''')

# Donut chart — simple, clean, conveys the key ratio at a glance
fig, ax = plt.subplots(figsize=(7, 7))
labels = [LABEL_BINARY[c] for c in df_binary['completed']]
colors = [PALETTE_BINARY[c] for c in df_binary['completed']]
wedges, texts, autotexts = ax.pie(
    df_binary['n'], labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75, textprops={'fontsize': 13},
)
# Create the donut hole
centre_circle = plt.Circle((0, 0), 0.50, fc='white')
ax.add_artist(centre_circle)
ax.set_title('Overall Completion Rate (binary)', fontsize=14, pad=20)
fig.tight_layout()
save_fig(fig, '01_outcome_distribution_binary')
plt.show()

# Print the headline number
completion_rate = df_binary.loc[df_binary['completed'] == 1, 'pct'].iloc[0]
print(f'\nHeadline: {completion_rate:.1f}% of enrollments result in completion.')

> **Interpretation:**
> - **Withdrawn** is the largest non-completion category. These students actively left — they are the primary target for retention interventions.
> - **Fail** represents students who stayed engaged but did not pass. They need a different type of support (academic, not motivational).
> - From a platform perspective: a significant share of students who enroll do **not** complete. This is the central problem this project investigates.
>
> The next sections will explore *who* these students are and *where* outcomes differ.

## 4. Outcome by Course

Before examining demographics or behavior, we check whether outcomes vary substantially **across courses**. If one course has a 70% completion rate and another 30%, then any population-level average masks important structural differences.

This is a preview for **BQ4** ("How do course characteristics affect retention?"), which Notebook 06 will analyze in depth.

In [ ]:
# --- Outcome proportions by module ---
# 100% stacked bar: normalizes for enrollment size, making courses comparable
df_by_module = execute_query('''
    SELECT
        code_module,
        final_result,
        COUNT(*) AS n
    FROM v_student_enriched
    GROUP BY code_module, final_result
    ORDER BY code_module, final_result
''')

# Pivot to get one column per outcome category
df_pivot = df_by_module.pivot(index='code_module', columns='final_result', values='n').fillna(0)
# Normalize each row to 100%
df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

# Reorder columns for visual consistency (positive outcomes first)
col_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
df_pct = df_pct[[c for c in col_order if c in df_pct.columns]]

# Sort modules by completion rate (Distinction + Pass) for easier reading
df_pct['_completion'] = df_pct.get('Distinction', 0) + df_pct.get('Pass', 0)
df_pct = df_pct.sort_values('_completion', ascending=True)
df_pct = df_pct.drop(columns='_completion')

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_pct.plot.barh(
    stacked=True, ax=ax,
    color=[PALETTE_OUTCOME[c] for c in df_pct.columns],
    edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('Percentage of enrollments')
ax.set_title('Outcome Distribution by Module (normalized to 100%)')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine()
fig.tight_layout()
save_fig(fig, '01_outcome_by_module')
plt.show()

> **Interpretation:**
> - Outcomes vary **substantially** across modules. This confirms that course-level factors matter and justifies BQ4.
> - Modules at the bottom of the chart have the highest combined Pass + Distinction rates; those at the top have the most withdrawals.
> - The Withdrawn segment (purple) varies across courses, suggesting that some course designs may be more prone to triggering early departures.
>
> **Implication:** Any retention analysis that ignores course identity risks [Simpson's paradox](https://en.wikipedia.org/wiki/Simpson%27s_paradox) — where an aggregate trend reverses within subgroups. We will keep course awareness throughout.

## 5. Demographic Profile

Understanding **who the students are** is essential before analyzing **what they did**. This section profiles the student population across key demographic variables and examines whether completion rates differ by group.

**Variables examined:**
- **Gender** — binary in OULAD (M/F)
- **Age band** — categorized as 0-35, 35-55, or 55<=
- **Highest education** — prior qualification level
- **IMD band** — Index of Multiple Deprivation (socio-economic indicator, UK-specific)
- **Numeric variables** — `num_of_prev_attempts` and `studied_credits`

**Important caveat:** Observing that a demographic group has a lower completion rate does **not** mean that demographic factor *causes* lower completion. Correlation is not causation. BQ3 will formally compare the explanatory power of demographics vs. behavior.

Each chart below shows the **population distribution** (bar height = number of enrollments) and the **completion rate** per group (annotated percentage). This dual view reveals both the group size and the outcome pattern simultaneously.

### 5a. Gender

In [ ]:
df_gender = query_demo_distribution('gender')
print(df_gender.to_string(index=False))
plot_demo_completion(df_gender, 'Gender Distribution & Completion Rate', '01_demo_gender')

### 5b. Age Band

In [ ]:
df_age = query_demo_distribution('age_band')
print(df_age.to_string(index=False))
plot_demo_completion(df_age, 'Age Band Distribution & Completion Rate', '01_demo_age_band')

### 5c. Highest Education

This variable captures the highest qualification the student had **before** enrolling. Categories range from 'No Formal quals' to 'Post Graduate Qualification'. We use horizontal bars because the category labels are long.

In [ ]:
df_edu = query_demo_distribution('highest_education')
print(df_edu.to_string(index=False))
plot_demo_completion(
    df_edu, 'Education Level Distribution & Completion Rate',
    '01_demo_education', horizontal=True,
)

### 5d. IMD Band (Index of Multiple Deprivation)

**What is IMD?** The Index of Multiple Deprivation is a UK government measure of relative deprivation for small areas. It combines indicators of income, employment, education, health, crime, housing, and environment. In OULAD:
- **0-10%** = most deprived areas (lowest decile)
- **90-100%** = least deprived areas (highest decile)

This is a socio-economic proxy, not an individual income measure. It tells us about the *area* where the student lives, not their personal finances.

> **Note:** IMD band has known missing values in OULAD. We will quantify these in the Data Quality section.

In [ ]:
df_imd = query_demo_distribution('imd_band')
print(df_imd.to_string(index=False))
plot_demo_completion(
    df_imd, 'IMD Band Distribution & Completion Rate',
    '01_demo_imd_band',
)

### 5e. Numeric Variables: Previous Attempts & Studied Credits

Two numeric demographic variables deserve attention:
- **`num_of_prev_attempts`**: how many times the student previously attempted this module (0 = first attempt)
- **`studied_credits`**: total credit load the student is carrying in this presentation

We use box plots split by outcome to compare distributions between Completed and Not completed groups.

In [ ]:
# --- Numeric demographics by outcome ---
df_numeric = execute_query('''
    SELECT
        num_of_prev_attempts,
        studied_credits,
        completed
    FROM v_student_enriched
''')
df_numeric['outcome'] = df_numeric['completed'].map(LABEL_BINARY)

fig, axes = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Previous attempts
sns.boxplot(
    data=df_numeric, x='outcome', y='num_of_prev_attempts',
    palette=PALETTE_BINARY_LABELS, ax=axes[0],
)
axes[0].set_title('Previous Attempts by Outcome')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of previous attempts')

# Studied credits
sns.boxplot(
    data=df_numeric, x='outcome', y='studied_credits',
    palette=PALETTE_BINARY_LABELS, ax=axes[1],
)
axes[1].set_title('Studied Credits by Outcome')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total studied credits')

sns.despine()
fig.suptitle('Numeric Demographics by Completion Outcome', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '01_demo_numeric_by_outcome')
plt.show()

### Demographic Summary

**Key observations from this section:**
- Completion rates **vary by demographic group**, but the magnitude of the differences varies.
- Education level and IMD band tend to show the largest spread in completion rates.
- Students with more previous attempts may have lower completion rates — but this could reflect inherent course difficulty rather than student ability.

> **Causation warning:** Observing that Group A completes at a higher rate than Group B does NOT mean that *being in Group A causes* higher completion. There are confounding factors (e.g., course choice, motivation, prior knowledge) that we cannot isolate from demographics alone. BQ3 (Notebook 05) will formally compare the explanatory power of demographics versus behavior — the answer has direct implications for where a platform should invest its resources.

## 6. Enrollment Patterns

When do students register relative to the course start?

In OULAD, `date_registration` is measured in **days relative to course start** (day 0). Negative values mean the student registered *before* the course officially began — this is common in university systems where enrollment opens weeks or months in advance.

**Why this matters:** Early registration may signal higher motivation or better planning. If early registrants complete at higher rates, this is a behavioral marker (though not necessarily a causal one) that could inform intervention timing.

In [ ]:
# --- Registration day distribution by outcome ---
df_reg = execute_query('''
    SELECT
        date_registration,
        completed
    FROM v_student_enriched
    WHERE date_registration IS NOT NULL
''')
df_reg['outcome'] = df_reg['completed'].map(LABEL_BINARY)

fig, ax = plt.subplots(figsize=FIG_SIZE)
# Overlapping histograms: one per outcome group
for outcome_val, label in LABEL_BINARY.items():
    subset = df_reg[df_reg['completed'] == outcome_val]
    ax.hist(
        subset['date_registration'], bins=50, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )

# Vertical line at day 0 (course start)
ax.axvline(x=0, color='black', linestyle='--', linewidth=1, label='Course start (day 0)')

ax.set_xlabel('Registration day (relative to course start)')
ax.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('When Do Students Register? (by completion outcome)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '01_enrollment_registration_day')
plt.show()

In [ ]:
# --- Registration day summary statistics by outcome ---
# PERCENTILE_CONT is the ANSI SQL standard for computing medians —
# MEDIAN() is a DuckDB shortcut that would break on BigQuery migration
df_reg_stats = execute_query('''
    SELECT
        CASE WHEN completed = 1 THEN 'Completed' ELSE 'Not completed' END AS outcome,
        COUNT(*)                        AS n,
        ROUND(AVG(date_registration), 1)    AS mean_day,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY date_registration), 1) AS median_day,
        MIN(date_registration)              AS min_day,
        MAX(date_registration)              AS max_day
    FROM v_student_enriched
    WHERE date_registration IS NOT NULL
    GROUP BY completed
    ORDER BY completed DESC
''')

print('=== Registration Day Statistics by Outcome ===\n')
df_reg_stats

> **Interpretation:**
> - Most students register **before** the course starts (negative registration day), which is expected.
> - If there is a difference between completers and non-completers in registration timing, it suggests that **enrollment timing** may be a weak early signal. However, this is observational — early registration could simply proxy for motivation or institutional support, not directly cause completion.
> - Students who register very late (positive registration day, i.e. after course start) may already be at a disadvantage due to missed content.

## 7. Course Landscape

We now shift from *who are the students* to *what are the courses*.

Each course-presentation has different design characteristics: length, assessment density, VLE resource diversity. Do any of these relate to completion rates?

**Caveat:** With only ~22 course-presentations, sample size is very small for correlation analysis. Patterns here are **suggestive**, not conclusive. BQ4 (Notebook 06) will analyze course effects more rigorously.

In [ ]:
# --- Course design vs. outcomes: 1x3 scatter plot ---
# Each dot is a course-presentation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Assessment density vs. completion rate
axes[0].scatter(
    df_courses['assessments_per_30_days'], df_courses['completion_rate_pct'],
    s=df_courses['n_enrolled'] / 5,  # bubble size proportional to enrollment
    alpha=0.7, color='#4C72B0', edgecolor='white',
)
axes[0].set_xlabel('Assessments per 30 days')
axes[0].set_ylabel('Completion rate (%)')
axes[0].set_title('Assessment Density vs. Completion')

# 2. VLE resources vs. completion rate
axes[1].scatter(
    df_courses['n_vle_resources'], df_courses['completion_rate_pct'],
    s=df_courses['n_enrolled'] / 5,
    alpha=0.7, color='#55A868', edgecolor='white',
)
axes[1].set_xlabel('Number of VLE resources')
axes[1].set_ylabel('Completion rate (%)')
axes[1].set_title('VLE Resources vs. Completion')

# 3. Course length vs. withdrawal rate
axes[2].scatter(
    df_courses['course_length_days'], df_courses['withdrawal_rate_pct'],
    s=df_courses['n_enrolled'] / 5,
    alpha=0.7, color='#C44E52', edgecolor='white',
)
axes[2].set_xlabel('Course length (days)')
axes[2].set_ylabel('Withdrawal rate (%)')
axes[2].set_title('Course Length vs. Withdrawal')

for ax in axes:
    sns.despine(ax=ax)

fig.suptitle('Course Design Characteristics vs. Student Outcomes\n(bubble size = enrollment)',
             fontsize=13, y=1.04)
fig.tight_layout()
save_fig(fig, '01_course_scatter_outcomes')
plt.show()

In [ ]:
# --- Correlation heatmap of course-level numeric features ---
# Which course characteristics move together?
cols_numeric = [
    'course_length_days', 'n_enrolled', 'completion_rate_pct',
    'withdrawal_rate_pct', 'n_assessments', 'n_vle_resources',
]
# Shorter labels for readability in the heatmap
label_map = {
    'course_length_days': 'Length (days)',
    'n_enrolled': 'Enrolled',
    'completion_rate_pct': 'Completion %',
    'withdrawal_rate_pct': 'Withdrawal %',
    'n_assessments': 'Assessments',
    'n_vle_resources': 'VLE resources',
}

df_corr = df_courses[cols_numeric].rename(columns=label_map).corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    df_corr, annot=True, fmt='.2f', cmap=PALETTE_SEQUENTIAL,
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    ax=ax,
)
ax.set_title('Correlation Between Course-Level Features')
fig.tight_layout()
save_fig(fig, '01_course_correlation_heatmap')
plt.show()

> **Interpretation:**
> - The scatter plots show whether course design features (assessment frequency, resource quantity, duration) have any visible relationship with outcomes. Bubble size encodes enrollment count, giving a sense of which courses contribute most data.
> - The correlation heatmap reveals **linear relationships** between course features. Strong correlations (close to +1 or -1) suggest features that move together; weak correlations (close to 0) suggest independence.
> - With only ~22 data points (course-presentations), **individual outliers can dominate** correlations. These results are exploratory hypotheses, not confirmed effects.
>
> BQ4 (Notebook 06) will analyze course effects in detail, controlling for student demographics.

## 8. Data Quality Assessment

**Why check data quality?**  
Even well-curated public datasets can have surprises: missing values, unexpected categories, outliers, or coverage gaps. Professional EDA always includes a quality check — *trust but verify*.

What we check:
1. **Missing values** — which columns have NULLs, and how many?
2. **Cardinality** — how many distinct values does each categorical column have?
3. **Range checks** — are numeric values within expected bounds?
4. **Coverage gaps** — do all views contain the expected number of students?

In [ ]:
# --- Missing value audit ---
# Count NULLs per column in v_student_enriched
df_nulls = execute_query('''
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END)              AS null_gender,
        SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END)              AS null_region,
        SUM(CASE WHEN highest_education IS NULL THEN 1 ELSE 0 END)   AS null_education,
        SUM(CASE WHEN imd_band IS NULL THEN 1 ELSE 0 END)            AS null_imd_band,
        SUM(CASE WHEN age_band IS NULL THEN 1 ELSE 0 END)            AS null_age_band,
        SUM(CASE WHEN disability IS NULL THEN 1 ELSE 0 END)          AS null_disability,
        SUM(CASE WHEN date_registration IS NULL THEN 1 ELSE 0 END)   AS null_registration,
        SUM(CASE WHEN final_result IS NULL THEN 1 ELSE 0 END)        AS null_final_result
    FROM v_student_enriched
''')

total = df_nulls['total_rows'].iloc[0]
# Transpose for readability: one row per column
null_cols = [c for c in df_nulls.columns if c.startswith('null_')]
null_summary = pd.DataFrame({
    'column': [c.replace('null_', '') for c in null_cols],
    'null_count': [int(df_nulls[c].iloc[0]) for c in null_cols],
})
null_summary['null_pct'] = (null_summary['null_count'] / total * 100).round(2)

print(f'=== Missing Value Audit ({total:,} total rows) ===\n')
null_summary

In [ ]:
# --- Cardinality check ---
# How many distinct values does each categorical column have?
df_card = execute_query('''
    SELECT
        COUNT(DISTINCT gender)              AS distinct_gender,
        COUNT(DISTINCT region)              AS distinct_region,
        COUNT(DISTINCT highest_education)   AS distinct_education,
        COUNT(DISTINCT imd_band)            AS distinct_imd,
        COUNT(DISTINCT age_band)            AS distinct_age,
        COUNT(DISTINCT disability)          AS distinct_disability,
        COUNT(DISTINCT final_result)        AS distinct_result,
        COUNT(DISTINCT code_module)         AS distinct_module
    FROM v_student_enriched
''')

print('=== Cardinality (distinct values per column) ===\n')
card_summary = df_card.T.reset_index()
card_summary.columns = ['column', 'n_distinct']
card_summary['column'] = card_summary['column'].str.replace('distinct_', '')
card_summary

In [ ]:
# --- Range checks for numeric columns ---
df_ranges = execute_query('''
    SELECT
        MIN(num_of_prev_attempts) AS min_prev_attempts,
        MAX(num_of_prev_attempts) AS max_prev_attempts,
        MIN(studied_credits)      AS min_credits,
        MAX(studied_credits)      AS max_credits,
        MIN(date_registration)    AS min_reg_day,
        MAX(date_registration)    AS max_reg_day,
        MIN(dropout_day)          AS min_dropout_day,
        MAX(dropout_day)          AS max_dropout_day
    FROM v_student_enriched
''')

print('=== Numeric Range Checks ===\n')
# Reshape for readability
ranges = {
    'num_of_prev_attempts': (df_ranges['min_prev_attempts'].iloc[0], df_ranges['max_prev_attempts'].iloc[0]),
    'studied_credits': (df_ranges['min_credits'].iloc[0], df_ranges['max_credits'].iloc[0]),
    'date_registration': (df_ranges['min_reg_day'].iloc[0], df_ranges['max_reg_day'].iloc[0]),
    'dropout_day': (df_ranges['min_dropout_day'].iloc[0], df_ranges['max_dropout_day'].iloc[0]),
}
pd.DataFrame(ranges, index=['min', 'max']).T

In [ ]:
# --- Coverage check: "ghost students" ---
# v_engagement_early only contains enrollments with at least one VLE click
# in days 0-28. Enrollments with ZERO activity are absent from that view.
# The gap between v_student_enriched and v_engagement_early = ghost students.
df_coverage = execute_query('''
    SELECT
        (SELECT COUNT(*) FROM v_student_enriched) AS n_total,
        (SELECT COUNT(*) FROM v_engagement_early) AS n_with_activity
''')

n_total = df_coverage['n_total'].iloc[0]
n_active = df_coverage['n_with_activity'].iloc[0]
n_ghosts = n_total - n_active
ghost_pct = 100 * n_ghosts / n_total

print('=== Engagement Coverage Check ===')
print(f'  Enrollments in v_student_enriched:    {n_total:>8,}')
print(f'  Enrollments in v_engagement_early:    {n_active:>8,}')
print(f'  "Ghost students" (zero activity):     {n_ghosts:>8,} ({ghost_pct:.1f}%)')
print(f'\n  → {n_ghosts:,} enrollments had zero VLE activity in the first 28 days.')
print('    These enrollments are ABSENT from v_engagement_early by design.')
print('    This is not a data quality issue — it is an analytical finding (BQ5 segment).')

> **Data quality verdict:**
> - **Missing values**: `imd_band` has known NULLs (documented in OULAD). Other columns are generally complete.
> - **Cardinality**: all categorical columns have the expected number of distinct values — no unexpected categories.
> - **Ranges**: numeric columns are within plausible bounds. Negative registration days are expected (pre-course enrollment). Dropout days are positive (relative to course start).
> - **Ghost students**: a measurable share of enrollments have zero VLE activity in the first 28 days. This is not missing data — these students simply never engaged with the platform. They represent a distinct segment for intervention (BQ5).
>
> **Conclusion:** The dataset is clean enough for analysis. The only caveat is `imd_band` nulls, which we handle by including the NULL category in analyses or excluding it where noted.

## 9. Engagement Baseline — Preview

Before closing this notebook, we take a **brief look** at early engagement. The full behavioral analysis is in Notebook 02 — here we just establish a baseline.

We use `v_engagement_early`, which aggregates clickstream activity for the first 28 days of each course. **Important:** this view only includes students with at least one click — the "ghost students" identified in Section 8 are not in this view.

In [ ]:
# --- Early engagement summary + outcome split ---
df_engage = execute_query('''
    SELECT
        ee.total_clicks_first_28,
        ee.active_days_first_28,
        se.completed
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
''')
df_engage['outcome'] = df_engage['completed'].map(LABEL_BINARY)

# Summary statistics
print('=== Early Engagement (first 28 days) ===\n')
print(df_engage.groupby('outcome')[['total_clicks_first_28', 'active_days_first_28']]
      .describe().round(1))

# Violin plot: total clicks by outcome
# Violins show the full distribution shape, not just quartiles
fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.violinplot(
    data=df_engage, x='outcome', y='total_clicks_first_28',
    palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
)
ax.set_xlabel('')
ax.set_ylabel('Total clicks in first 28 days')
ax.set_title('Early Engagement by Completion Outcome')
sns.despine()
fig.tight_layout()
save_fig(fig, '01_engagement_preview_by_outcome')
plt.show()

print('\nNote: "ghost students" (zero activity) are excluded from this view.')
print('The full engagement analysis is in Notebook 02.')

## 10. Key Takeaways and Next Steps

### What we learned

1. **Scale**: The OULAD dataset contains ~32K enrollments across ~28K unique students in 22 course-presentations. The unit of analysis is the enrollment, not the student.

2. **Completion challenge**: A significant share of enrollments do not result in completion. Withdrawn students (who actively left) outnumber those who stayed but failed — suggesting that **retention**, not academic support, is the primary lever.

3. **Course variation**: Completion rates vary substantially across modules. Any analysis that ignores course identity risks Simpson's paradox.

4. **Demographics matter — but how much?** Completion rates differ by education level, IMD band, and other demographics. BQ3 will test whether these differences are large enough to be actionable, or whether behavior is a stronger signal.

5. **Early engagement signal**: Even in this brief preview, completed students show notably higher VLE activity in the first 28 days. BQ2 will quantify the predictive strength of these early signals.

6. **Ghost students**: A measurable segment of students enroll but never interact with the VLE. They are a natural target for early intervention (BQ5).

7. **Data quality**: The dataset is clean. Known caveats (IMD band nulls, ghost student coverage) are documented and accounted for.

### What comes next

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **02** | — | EDA: engagement patterns (daily clickstream, temporal trends) |
| **03** | BQ1 | Where and when do students drop out? |
| **04** | BQ2 | Which early behavioral signals predict drop-out? |
| **05** | BQ3 | Demographics vs. behavior — what predicts outcome more? |
| **06** | BQ4 | How do course characteristics affect retention? |
| **07** | BQ5 | Top 3 actionable interventions |

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **Preview finding:** Completed students have visibly **higher early engagement** than non-completers — both in total clicks and in the shape of the distribution (longer tail towards high engagement).
>
> This is a **descriptive observation**, not a causal claim. It will be quantified with statistical tests (t-test, effect size) in Notebook 04 (BQ2: early behavioral signals).
>
> Continue to **Notebook 02** (`02_eda_engagement_patterns.ipynb`) for the full behavioral analysis.